##**Iris Dataset Classification Analysis**
##**Course**: STQD6324 Data Management
##**Semester**: Semester 2 2025/2026

####This Notebook uses Spark MLlib to perform multi-class classification tasks on the classic Iris dataset, including data preprocessing, training and tuning of three classification models, and comparative analysis of model performance.

#1. Environment Preparation First
####confirm that PySpark is installed, and then import all the libraries you need at once.

In [1]:
!pip install pyspark -q

from sklearn.datasets import load_iris
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("All libraries have been imported.")

All libraries have been imported.


##2. Start a Spark session
###Run PySpark on Colab in local mode.

In [2]:
spark = SparkSession.builder \
    .appName("Iris_Classification") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version：", spark.version)

Spark version： 4.0.2


##3. Load the Iris dataset

###The Iris dataset contains 150 records with 4 features (calyx length and width, petal length and width) and 3 categories (Setosa / Versicolor / Virginica).

###Here, the data is obtained using sklearn and then converted into a Spark DataFrame for subsequent processing using the Spark API.

In [3]:
# Use sklearn to get the data, saving you the trouble of manually downloading CSV files.
iris = load_iris()
pdf = pd.DataFrame(iris.data, columns=iris.feature_names)
pdf["label"] = iris.target

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

print(f"Dataset size：{df.count()} rows，{len(df.columns)} columns")
print("Sample size of each category：")
df.groupBy("label").count().orderBy("label").show()

df.show(5)

Dataset size：150 rows，5 columns
Sample size of each category：
+-----+-----+
|label|count|
+-----+-----+
|    0|   50|
|    1|   50|
|    2|   50|
+-----+-----+

+-----------------+----------------+-----------------+----------------+-----+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|label|
+-----------------+----------------+-----------------+----------------+-----+
|              5.1|             3.5|              1.4|             0.2|    0|
|              4.9|             3.0|              1.4|             0.2|    0|
|              4.7|             3.2|              1.3|             0.2|    0|
|              4.6|             3.1|              1.5|             0.2|    0|
|              5.0|             3.6|              1.4|             0.2|    0|
+-----------------+----------------+-----------------+----------------+-----+
only showing top 5 rows


##4. Data Preprocessing

###Spark MLlib requires all feature columns to be merged into a single vector column, which is done using `VectorAssembler`.

###Iris data is relatively clean, with no missing values, and therefore does not require additional imputation or normalization steps.

In [4]:
feature_cols = iris.feature_names

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
df = assembler.transform(df)

# Only feature vectors and label columns are retained.
df = df.select("features", "label")

df.show(5, truncate=False)

+-----------------+-----+
|features         |label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|0    |
|[4.9,3.0,1.4,0.2]|0    |
|[4.7,3.2,1.3,0.2]|0    |
|[4.6,3.1,1.5,0.2]|0    |
|[5.0,3.6,1.4,0.2]|0    |
+-----------------+-----+
only showing top 5 rows


##5. Split the training and testing sets.

###Randomly split the sets in a 7:3 ratio, using a fixed random `seed=42` for easy reproduction.

###Approximately 105 of the 150 data points are used for training, and 45 for testing.

In [5]:
train, test = df.randomSplit([0.7, 0.3], seed=42)

print(f"training set：{train.count()} strips")
print(f"test set：{test.count()} strips")

train.show(5, truncate=False)
test.show(5, truncate=False)

training set：94 strips
test set：56 strips
+-----------------+-----+
|features         |label|
+-----------------+-----+
|[4.3,3.0,1.1,0.1]|0    |
|[4.4,2.9,1.4,0.2]|0    |
|[4.4,3.2,1.3,0.2]|0    |
|[4.5,2.3,1.3,0.3]|0    |
|[4.6,3.1,1.5,0.2]|0    |
+-----------------+-----+
only showing top 5 rows
+-----------------+-----+
|features         |label|
+-----------------+-----+
|[4.4,3.0,1.3,0.2]|0    |
|[4.6,3.2,1.4,0.2]|0    |
|[4.6,3.6,1.0,0.2]|0    |
|[4.7,3.2,1.3,0.2]|0    |
|[4.8,3.1,1.6,0.2]|0    |
+-----------------+-----+
only showing top 5 rows


##6. Define the evaluator.
###All three models share the same evaluator object, which will also be used in cross-validation.

###The main evaluation metric is Accuracy first; F1, Precision, and Recall will be calculated separately during the final comparison.

In [6]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
evaluator

MulticlassClassificationEvaluator_870824f50ee7

##7. Model 1: Logistic Regression
###Logistic regression is a linear classification model. It trains quickly, has strong interpretability, and is suitable as a baseline solution.

##**Tuning Parameters:**

- `regParam`: L2 regularization strength to prevent overfitting. Values `​​[0.01, 0.05, 0.1]` cover a range from weak to moderate regularization.

- `maxIter`: Maximum number of iterations. For small Iris datasets, 100 iterations are sufficient for convergence.

- Cross-validation folds: 5 folds (more folds for smaller datasets result in greater stability).

In [7]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    family="multinomial"   # Three-category classification requires the use of multinomial.
)

paramGrid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.05, 0.1])
    .addGrid(lr.maxIter, [100])
    .build()
)

cv_lr = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

print("train Logistic Regression...")
model_lr = cv_lr.fit(train)
pred_lr = model_lr.transform(test)

# Optimal hyperparameters
best_lr = model_lr.bestModel
print(f"optimal regParam：{best_lr._java_obj.getRegParam()}")

# Each indicator
acc_lr       = evaluator.evaluate(pred_lr)
f1_lr        = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_lr)
precision_lr = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(pred_lr)
recall_lr    = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(pred_lr)

print(f"Accuracy={acc_lr:.4f}  Precision={precision_lr:.4f}  Recall={recall_lr:.4f}  F1={f1_lr:.4f}")

train Logistic Regression...
optimal regParam：0.01
Accuracy=0.9643  Precision=0.9694  Recall=0.9643  F1=0.9647


##8. Model 2: Decision Tree
##Decision trees classify data by recursively partitioning the feature space, providing intuitive results and allowing direct visualization of the rules.

###They are insensitive to feature scale and do not require normalization.

**Optimization parameters:**

- `maxDepth:` The maximum depth of the tree. Too shallow, it underfits; too deep, it overfits. Values `​​[3, 5, 7]` cover different complexities.

- `minInstancesPerNode:` The minimum number of samples per leaf node, controlling overfitting. Values `​​[1, 2]`.

In [8]:
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

paramGrid_dt = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 7])
    .addGrid(dt.minInstancesPerNode, [1, 2])
    .build()
)

cv_dt = CrossValidator(
    estimator=dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

print("train Decision Tree...")
model_dt = cv_dt.fit(train)
pred_dt = model_dt.transform(test)

best_dt = model_dt.bestModel
print(f"optimal maxDepth：{best_dt.depth}")

acc_dt       = evaluator.evaluate(pred_dt)
f1_dt        = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_dt)
precision_dt = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(pred_dt)
recall_dt    = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(pred_dt)

print(f"Accuracy={acc_dt:.4f}  Precision={precision_dt:.4f}  Recall={recall_dt:.4f}  F1={f1_dt:.4f}")

train Decision Tree...
optimal maxDepth：5
Accuracy=0.9821  Precision=0.9835  Recall=0.9821  F1=0.9823


##9. Model 3: Random Forest
###Random Forest is an ensemble version of decision trees. It reduces variance through bootstrap sampling and random feature selection, and is generally more stable than a single decision tree.

**Tuning parameters:**

- `numTrees`: The number of trees. More trees result in greater stability but slower training. Values ​​are `[20, 50, 100]`.

- `maxDepth`: The depth of a single tree. Values ​​are `[3, 5]`.

- `featureSubsetStrategy`: The proportion of features randomly selected during each split. The default value is `"auto"`, i.e., sqrt(numFeatures).

In [9]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

paramGrid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50, 100])
    .addGrid(rf.maxDepth, [3, 5])
    .build()
)

cv_rf = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

print("train Random Forest...")
model_rf = cv_rf.fit(train)
pred_rf = model_rf.transform(test)

best_rf = model_rf.bestModel
print(f"optimal numTrees：{best_rf.getNumTrees}，maxDepth：{best_rf.getMaxDepth()}")

acc_rf       = evaluator.evaluate(pred_rf)
f1_rf        = MulticlassClassificationEvaluator(metricName="f1").evaluate(pred_rf)
precision_rf = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(pred_rf)
recall_rf    = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(pred_rf)

print(f"Accuracy={acc_rf:.4f}  Precision={precision_rf:.4f}  Recall={recall_rf:.4f}  F1={f1_rf:.4f}")

train Random Forest...
optimal numTrees：50，maxDepth：5
Accuracy=0.9821  Precision=0.9835  Recall=0.9821  F1=0.9823


##10. Model Performance Comparison

In [10]:
results = {
    "Logistic Regression": {
        "Accuracy": acc_lr, "Precision": precision_lr,
        "Recall": recall_lr, "F1": f1_lr
    },
    "Decision Tree": {
        "Accuracy": acc_dt, "Precision": precision_dt,
        "Recall": recall_dt, "F1": f1_dt
    },
    "Random Forest": {
        "Accuracy": acc_rf, "Precision": precision_rf,
        "Recall": recall_rf, "F1": f1_rf
    },
}

result_df = pd.DataFrame(results).T.round(4)
print(result_df.to_string())

best_name = result_df["Accuracy"].idxmax()
print(f"\nOptimal Model（according to Accuracy）：{best_name}  →  {result_df.loc[best_name, 'Accuracy']:.4f}")

                     Accuracy  Precision  Recall      F1
Logistic Regression    0.9643     0.9694  0.9643  0.9647
Decision Tree          0.9821     0.9835  0.9821  0.9823
Random Forest          0.9821     0.9835  0.9821  0.9823

Optimal Model（according to Accuracy）：Decision Tree  →  0.9821


## 11. Comparative Analysis and Conclusions
##  Model Performance Analysis
**Logistic Regression:**
Performs well on Iris because the three classes are basically linearly separable in the feature space (Setosa is very well separated from the other two classes). The main limitation is a slightly weaker fit to the boundary between Versicolor and Virginica, as these two classes have some overlap in feature distribution, making the linear decision boundary less flexible.

**Decision Tree:**
Tree structures can directly capture feature threshold rules, making them suitable for data like Iris where "classification can be achieved with a few if-else statements." However, a single tree is significantly affected by random splitting of the training set, and results may fluctuate under different random seeds.

**Random Forest:**
After ensembled multiple trees, variance decreases significantly, making it the most stable of the three in most cases. For Iris with its small dataset, 100 trees are sufficient. If the data is more complex (severe class overlap, high noise), the advantages of random forest over a single decision tree become more pronounced.

### Optimal Model Selection:
Considering all four metrics, **random forest** typically performs best in accuracy and F1 score. Its main advantages are:

- It is insensitive to hyperparameters and has strong generalization ability.
- It comes with a built-in feature importance score, providing some interpretability.
- It does not require feature normalization.

If the production environment has requirements for **model size** **and inference speed**, logistic regression is a lighter alternative with similar accuracy but much lower deployment costs.


### Meaning of the four indicators + evaluation criteria
**Accuracy:** The percentage of correctly predicted samples out of all samples. A higher value indicates more accurate overall predictions.

**Precision:** The percentage of samples predicted as belonging to a certain class that actually belong to that class. A higher percentage indicates fewer misclassifications.

**Recall:** The percentage of samples that actually belong to that class that are successfully identified. A higher percentage indicates fewer missed detections of true samples.

**F1-Score:** The harmonic mean of precision and recall. A better balance between the two is better.

### Meaning of Values
A value closer to 1 (1.0) = better model performance; closer to 0 = essentially ineffective model.
A good model typically has all four metrics ≥ 0.85.

### Rules for Selecting the Best Model (Standard Answer)
Prioritize Accuracy: Highest overall accuracy indicates best overall performance. When accuracy is similar, consider the F1 score. A higher F1 score indicates a better balance between precision and recall.

### Business-oriented selection:
To avoid misclassification: Choose models with high precision.
To avoid missed samples: Choose models with high recall.
General conclusion: The optimal model is one where all four metrics are high.

##12. Random Forest Feature Importance
###Let's look at which features contribute the most to classification.

In [11]:
importances = best_rf.featureImportances.toArray()
feat_names  = iris.feature_names

feat_df = pd.DataFrame({
    "feature":    feat_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(feat_df.to_string(index=False))
# Petal length and width are usually the first two factors to consider, which aligns with intuition.

          feature  importance
 petal width (cm)    0.449844
petal length (cm)    0.432729
sepal length (cm)    0.104165
 sepal width (cm)    0.013262


##13. Test Set Prediction Examples
###The optimal model (random forest) is used to make predictions on the test set, and some results are shown.

In [12]:
# Replacing the numeric labels with actual class names makes it more intuitive.
label_map = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}

pred_show = pred_rf.select("features", "label", "prediction").limit(10).toPandas()
pred_show["label_name"]      = pred_show["label"].map(label_map)
pred_show["prediction_name"] = pred_show["prediction"].map(label_map)
pred_show["correct"]         = pred_show["label"] == pred_show["prediction"]

print(pred_show[["label_name", "prediction_name", "correct"]].to_string(index=False))

label_name prediction_name  correct
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
    Setosa          Setosa     True
Versicolor      Versicolor     True
    Setosa          Setosa     True
